# Metaphors
- LDA or similar on each domain cluster to get key words
- include source domains and KI_sentences (do some filtering with knowledge from previous experiments?)

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
!pip install -q transformers datasets evaluate shap torch tqdm scikit-learn
!python -m spacy download de_core_news_lg

In [ ]:
# ── Upload data ───────────────────────────────────────────────────────────────
# Upload metaphors_data.zip when prompted, then run this cell.
#
# Alternatively, if the zip is already on Google Drive, mount it first:
#   from google.colab import drive
#   drive.mount('/content/drive')
# and set data_zip_path to the correct Drive path.

import zipfile, os
from google.colab import files

uploaded = files.upload()          # select metaphors_data.zip
data_zip_path = list(uploaded.keys())[0]

os.makedirs('generated_data/AI_sentences',    exist_ok=True)
os.makedirs('generated_data/Anthrop_sentences', exist_ok=True)

with zipfile.ZipFile(data_zip_path, 'r') as z:
    # all four files are at the root of the zip (-j flag was used)
    z.extract('Leipzig_KI_sentences_all.json', 'generated_data/AI_sentences/')
    z.extract('KI_sentences_translated.csv',   'generated_data/AI_sentences/')
    z.extract('all_anthro_ki.json',            'generated_data/Anthrop_sentences/')
    z.extract('domains_all.csv',               'generated_data/')

print('Data extracted.')

In [ ]:
from pprint import pprint
import os
import re
import json
import ast
import copy
import dataclasses
import string

import pandas as pd
import numpy as np
import tqdm
import shap
import matplotlib.pyplot as plt

import spacy
nlp = spacy.load('de_core_news_lg')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score,
)

from datasets import load_dataset
from datasets import Dataset as HFDataset

from transformers import (
    TrainingArguments, AutoTokenizer, Trainer,
    EarlyStoppingCallback, DataCollatorWithPadding,
    DataCollatorForLanguageModeling, AutoModel,
    AutoModelForMaskedLM, get_linear_schedule_with_warmup,
    AutoConfig,
)
from transformers.trainer_utils import get_last_checkpoint
from evaluate import load

import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device} device')

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

def flatten(xss):
    return [x for xs in xss for x in xs]

plots_dir    = 'plots/'
metaphors_dir = 'Metaphor_model/'
os.makedirs(metaphors_dir, exist_ok=True)
os.makedirs(plots_dir,     exist_ok=True)

In [ ]:
# Datasets
# load docs (KI sentences) to analyse
with open('generated_data/AI_sentences/Leipzig_KI_sentences_all.json', 'r') as f:
    leipzig_data = json.load(f)
leipzig_data = pd.DataFrame(leipzig_data)
leipzig_data = (
    leipzig_data
    .reset_index()
    .rename(columns={'index': 'id'})
    .melt(id_vars=['id'], var_name='year', value_name='sentence')
    .dropna(subset=['sentence'])
)
taz24_data = pd.read_csv('generated_data/AI_sentences/KI_sentences_translated.csv').rename(
    columns={'original': 'sentence'}
)
print('leipzig:', leipzig_data.columns.tolist(), leipzig_data.shape,
      '\ntaz24:',   taz24_data.columns.tolist(), taz24_data.shape)

# load docs (anthropomorphic KI sentences) to analyse
with open('generated_data/Anthrop_sentences/all_anthro_ki.json', 'r') as f:
    anthro_data = json.load(f)

    anthro_dfs = []
    for rel in ['sb', 'ob', 'v']:
        rel_data = anthro_data.get(rel, [])
        if isinstance(rel_data, list) and len(rel_data) == 1 and isinstance(rel_data[0], dict):
            rel_data = rel_data[0]

        if isinstance(rel_data, dict):
            df_rel = (
                pd.DataFrame.from_dict(rel_data, orient='index')
                .rename_axis('id')
                .reset_index()
                .assign(relation_type=rel)
            )
        else:
            df_rel = pd.DataFrame(
                columns=['id', 'type', 'verb', 'dependent', 'dep_label',
                         'verb_pos', 'dep_pos', 'sentence']
            )
            df_rel['relation_type'] = rel

        anthro_dfs.append(df_rel)

    anthro_data = pd.concat(anthro_dfs, ignore_index=True)
    anthro_data = anthro_data.drop(columns=['type', 'dep_label', 'verb_pos'])

print('anthro:', anthro_data.columns.tolist(), anthro_data.shape)

# domains
domains    = pd.read_csv('generated_data/domains_all.csv', sep=',')
domains_grouped = domains.groupby('domain')
print('domains:', domains.columns.tolist(), domains.shape)

datasets_all = {
    'leipzig': leipzig_data,
    'taz24':   taz24_data,
    'anthro':  anthro_data,
    'domains': domains,
}

In [ ]:
def add_pos_tags(df):
    try:
        pos_tags = []
        for sentence in tqdm.tqdm(df['sentence']):
            doc = nlp(sentence)
            pos_tags.append([token.pos_ for token in doc])
        df.insert(1, 'pos_tags', pos_tags)
    except Exception as e:
        print(f'Error processing DataFrame: {e}')
    return df

In [ ]:
def preprocess_df(df, term_re):
    try:
        ki_terms = []
        for sentence in df['sentence']:
            match = re.search(term_re, sentence)
            ki_terms.append(match.group(0) if match else None)
        df.insert(1, 'term',   ki_terms)
        df.insert(1, 'domain', 'KI')
        df = add_pos_tags(df)
    except Exception as e:
        print(f'Error processing DataFrame: {e}')
    return df

regex_ki = (
    r'\bki\b|\bkünstliche\sintelligenz\b|\bchatgpt\b|\bai\b|\bcopilot\b'
    r'|\bgemini\b|\bchatbot\b|\bllm\b|\bchat-gpt\b|\bsprachmodell\b'
    r'|\bki-modell\b|\bki-system\b|\bki-tool\b|\bartificial\sintelligence\b'
    r'|\bki-anwendung\b|\bpalm\b|\blms\b|\bllama\b|\btransformer\b'
    r'|\bsprachmodelle\b|\bgpt\b|\bplms\b|\bvortrainierte\ssprachmodelle\b'
    r'|\bgpt-2\b|\bxlnet\b|\bgroße\ssprachmodelle\b|\bllms\b|\bgpt-3\b'
    r'|\bstiftungsmodell\b|\bgpt-neo\b|\bgpt-j\b|\bgpt-4\b'
)

for name, df in list(datasets_all.items())[:3]:
    preprocess_df(df, regex_ki)

# add POS tags for domains
df = datasets_all['domains']
df = add_pos_tags(df)
datasets_all['domains'] = df

In [ ]:
# split datasets into train / val / test
leipzig_train, leipzig_test = train_test_split(leipzig_data, test_size=0.2, random_state=42)
taz24_train,   taz24_test   = train_test_split(taz24_data,   test_size=0.2, random_state=42)
anthro_train,  anthro_test  = train_test_split(anthro_data,  test_size=0.2, random_state=42)

domain_train_test = {}
for d, group in domains_grouped:
    domain_train_test[d] = train_test_split(group, test_size=0.4, random_state=42)

leipzig_test, leipzig_val = train_test_split(leipzig_test, test_size=0.5, random_state=42)
taz24_test,   taz24_val   = train_test_split(taz24_test,   test_size=0.5, random_state=42)
anthro_test,  anthro_val  = train_test_split(anthro_test,  test_size=0.5, random_state=42)

domain_train_val_test = {}
for d, (train, test) in domain_train_test.items():
    test, val = train_test_split(test, test_size=0.5, random_state=42)
    domain_train_val_test[d] = (train, val, test)

In [ ]:
split_dir = 'generated_data/train_test_splits'
os.makedirs(split_dir, exist_ok=True)

def save_and_load_split(name, train_df, test_df, val_df):
    train_path = os.path.join(split_dir, f'{name}_train.csv')
    test_path  = os.path.join(split_dir, f'{name}_test.csv')
    val_path   = os.path.join(split_dir, f'{name}_val.csv')
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path,   index=False)
    val_df.to_csv(val_path,     index=False)
    return load_dataset('csv', data_files={
        'train': train_path, 'test': test_path, 'validation': val_path
    })

datasets = {
    'leipzig': save_and_load_split('leipzig', leipzig_train, leipzig_test, leipzig_val),
    'taz24':   save_and_load_split('taz24',   taz24_train,   taz24_test,   taz24_val),
    'anthro':  save_and_load_split('anthro',  anthro_train,  anthro_test,  anthro_val),
}

domain_datasets = {
    domain: save_and_load_split(f'domain_{domain}', train, test, val)
    for domain, (train, val, test) in domain_train_val_test.items()
}

datasets.update(domain_datasets)

## Using BERT

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-german-cased')

def tokenize_function(examples):
    return tokenizer(examples['sentence'], padding='max_length',
                     truncation=True, max_length=128)

for name, dataset in datasets.items():
    print(f'Tokenizing {name} dataset...')
    tokenized_datasets = dataset.map(tokenize_function, batched=True)

print(tokenized_datasets['train'][0])

### Model architecture and training
Loosely based on BasicBert and MelBert.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-german-cased')

training_args = TrainingArguments(
    output_dir=f'{metaphors_dir}/results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    save_safetensors=False,
    logging_dir=f'{metaphors_dir}/logs',
    logging_steps=100,
    fp16=(device == 'cuda'),
)

metric = load('f1')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
class _SafeMLMCollator(DataCollatorForLanguageModeling):
    """Guarantees ≥1 masked token per sequence (prevents 0/0 loss edge case)."""
    def torch_call(self, examples):
        batch     = super().torch_call(examples)
        labels    = batch['labels']
        input_ids = batch['input_ids']
        special   = set(self.tokenizer.all_special_ids)
        pad       = self.tokenizer.pad_token_id
        for i in range(labels.shape[0]):
            if (labels[i] != -100).sum() == 0:
                for j in range(1, labels.shape[1]):
                    tok = input_ids[i, j].item()
                    if tok != pad and tok not in special:
                        labels[i, j]    = tok
                        input_ids[i, j] = self.tokenizer.mask_token_id
                        break
        batch['labels']    = labels
        batch['input_ids'] = input_ids
        return batch


def train_domain_model(domain_dataset, base_model, tokenizer, training_args,
                       validation_dataset, domain_name='domain'):
    """Fine-tunes an independent copy of base_model on a domain corpus via MLM."""
    model = copy.deepcopy(base_model)

    if 'input_ids' not in domain_dataset.column_names:
        def _tok(examples):
            return tokenizer(examples['sentence'], padding='max_length',
                             truncation=True, max_length=128)
        keep = {'input_ids', 'attention_mask', 'token_type_ids'}
        domain_dataset = domain_dataset.map(
            _tok, batched=True,
            remove_columns=[c for c in domain_dataset.column_names if c not in keep],
        )
        validation_dataset = validation_dataset.map(
            _tok, batched=True,
            remove_columns=[c for c in validation_dataset.column_names if c not in keep],
        )

    steps_per_epoch = max(1, len(domain_dataset) // training_args.per_device_train_batch_size)
    domain_args = dataclasses.replace(
        training_args,
        output_dir=f'{training_args.output_dir}/{domain_name}',
        logging_steps=max(1, steps_per_epoch // 4),
        warmup_ratio=0.1,
        fp16=(device == 'cuda'),
        bf16=False,
    )

    collator = _SafeMLMCollator(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
    trainer  = Trainer(
        model=model,
        args=domain_args,
        train_dataset=domain_dataset,
        data_collator=collator,
        eval_dataset=validation_dataset,
    )
    trainer.train(resume_from_checkpoint=False)
    return model, trainer.state.log_history

In [ ]:
base_bert = AutoModelForMaskedLM.from_pretrained('google-bert/bert-base-german-cased')

domain_models       = {}
domain_log_histories = {}

for domain, group_df in domains_grouped:
    save_dir  = f"{training_args.output_dir}/{domain}"
    last_ckpt = get_last_checkpoint(save_dir) if os.path.isdir(save_dir) else None

    if last_ckpt:
        print(f'Loading saved domain model: {domain}  ({last_ckpt})')
        domain_models[domain] = AutoModelForMaskedLM.from_pretrained(last_ckpt)
        state_path = os.path.join(save_dir, 'trainer_state.json')
        if os.path.isfile(state_path):
            with open(state_path) as f:
                domain_log_histories[domain] = json.load(f)['log_history']
        else:
            domain_log_histories[domain] = None
    else:
        sentences = group_df[['sentence']].dropna().reset_index(drop=True)
        train_df, val_df = train_test_split(sentences, test_size=0.2, random_state=42)

        train_hf = HFDataset.from_pandas(train_df.reset_index(drop=True))
        val_hf   = HFDataset.from_pandas(val_df.reset_index(drop=True))

        print(f'Training domain model: {domain}  (train={len(train_hf)}, val={len(val_hf)})')
        domain_models[domain], domain_log_histories[domain] = train_domain_model(
            train_hf, base_bert, tokenizer, training_args,
            val_hf, domain_name=domain,
        )

print(f'\nLoaded/trained {len(domain_models)} domain models: {list(domain_models.keys())}')

In [ ]:
colors = plt.cm.tab20.colors
fig, (ax_train, ax_val) = plt.subplots(1, 2, figsize=(14, 5))

for i, (domain, log_history) in enumerate(domain_log_histories.items()):
    if not log_history:
        continue
    color = colors[i % len(colors)]

    train_entries = [e for e in log_history if 'loss' in e and 'eval_loss' not in e]
    eval_entries  = [e for e in log_history if 'eval_loss' in e]

    if train_entries:
        ax_train.plot([e['epoch'] for e in train_entries],
                      [e['loss']  for e in train_entries],
                      color=color, label=domain, alpha=0.85)
    if eval_entries:
        ax_val.plot([e['epoch']     for e in eval_entries],
                    [e['eval_loss'] for e in eval_entries],
                    color=color, label=domain, marker='o', alpha=0.85)

for ax, title in zip((ax_train, ax_val), ('Train Loss', 'Validation Loss')):
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(title)
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{plots_dir}/domain_losses.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# MetaphorBERT: Siamese BERT for Metaphor Identification
# ============================================================

_GERMAN_POS = sorted({
    tag
    for df in datasets_all.values()
    if 'pos_tags' in df.columns
    for tags in df['pos_tags'].dropna()
    for tag in (tags if isinstance(tags, list) else [])
})
POS_SPECIAL_TOKENS = [f'[POS_{t}]' for t in _GERMAN_POS]
print(f'POS tags ({len(_GERMAN_POS)}): {_GERMAN_POS}')


def add_pos_tokens(tok):
    tok.add_special_tokens({'additional_special_tokens': POS_SPECIAL_TOKENS})
    return tok


class MetaphorDataset(Dataset):
    """
    Word-level dataset: one sample per (sentence, target word).
    Sentence encoder input: [CLS] sentence [SEP] [POS_TAG]
    Word encoder input:     target word only (domain embedding)
    """

    def __init__(self, df, tokenizer, domain_names,
                 max_sent_len=128, max_word_len=16, label_col='label'):
        self.tokenizer    = tokenizer
        self.domain_names = domain_names
        self.max_sent_len = max_sent_len
        self.max_word_len = max_word_len
        self.samples      = []

        for _, row in df.iterrows():
            sentence   = str(row['sentence'])
            pos_tags   = row.get('pos_tags', [])
            raw_labels = row.get(label_col, None)

            words = sentence.split()
            if not isinstance(pos_tags, list) or len(pos_tags) != len(words):
                pos_tags = ['X'] * len(words)
            labels = raw_labels if isinstance(raw_labels, list) and len(raw_labels) == len(words) \
                     else [0] * len(words)

            for w_idx, (word, pos, label) in enumerate(zip(words, pos_tags, labels)):
                self.samples.append({
                    'sentence': sentence, 'word': word,
                    'word_idx': w_idx, 'pos': pos, 'label': int(label),
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        pos_tok = f"[POS_{s['pos']}]" if f"[POS_{s['pos']}]" in \
                  self.tokenizer.additional_special_tokens else '[UNK]'

        enc = self.tokenizer(
            s['sentence'], pos_tok,
            max_length=self.max_sent_len, padding='max_length',
            truncation=True, return_tensors='pt',
        )
        input_ids      = enc['input_ids'].squeeze(0)
        attention_mask = enc['attention_mask'].squeeze(0)
        token_type_ids = enc.get(
            'token_type_ids',
            torch.zeros(self.max_sent_len, dtype=torch.long)
        ).squeeze(0)

        target_mask = self._build_target_mask(s['sentence'], s['word_idx'], input_ids)

        dom_ids, dom_masks = {}, {}
        for domain in self.domain_names:
            we = self.tokenizer(
                s['word'], max_length=self.max_word_len, padding='max_length',
                truncation=True, return_tensors='pt',
            )
            dom_ids[domain]   = we['input_ids'].squeeze(0)
            dom_masks[domain] = we['attention_mask'].squeeze(0)

        return {
            'input_ids': input_ids, 'attention_mask': attention_mask,
            'token_type_ids': token_type_ids, 'target_mask': target_mask,
            'domain_input_ids': dom_ids, 'domain_attention_masks': dom_masks,
            'label': torch.tensor(s['label'], dtype=torch.long),
        }

    def _build_target_mask(self, sentence, word_idx, input_ids):
        mask  = torch.zeros(len(input_ids), dtype=torch.float)
        words = sentence.split()
        pos   = 1  # skip [CLS]
        for i, word in enumerate(words):
            subwords = self.tokenizer.tokenize(word)
            n = len(subwords)
            if i == word_idx:
                for j in range(n):
                    if pos + j < len(input_ids):
                        mask[pos + j] = 1.0
                break
            pos += n
        return mask


def collate_metaphor(batch):
    keys = ['input_ids', 'attention_mask', 'token_type_ids', 'target_mask', 'label']
    out  = {k: torch.stack([b[k] for b in batch]) for k in keys}
    domains = list(batch[0]['domain_input_ids'].keys())
    out['domain_input_ids']        = {d: torch.stack([b['domain_input_ids'][d]        for b in batch]) for d in domains}
    out['domain_attention_masks']  = {d: torch.stack([b['domain_attention_masks'][d]  for b in batch]) for d in domains}
    return out


# ── Model ─────────────────────────────────────────────────────────────────────

class LatentInteraction(nn.Module):
    """Bridge the semantic gap between basic and contextual embeddings."""
    def __init__(self, hidden_size, out_size):
        super().__init__()
        self.linear = nn.Linear(hidden_size * 2, out_size)

    def forward(self, basic, ctx):
        return F.gelu(self.linear(torch.cat([basic, ctx], dim=-1)))


class MetaphorBERT(nn.Module):
    """
    Siamese BERT for per-word metaphor classification.

    Sentence encoder: [CLS] sentence [SEP] [POS_TAG] → contextual word embedding
    Word encoder (domain-fine-tuned BERT): target word → basic embedding
    Latent interaction per domain → concatenated → binary classifier
    Also returns which domain has the highest basic↔contextual distance.
    """

    def __init__(self, sentence_encoder, domain_encoders, bert_config,
                 num_labels=2, dropout=0.2, interaction_hidden=256):
        super().__init__()
        self.sentence_encoder = sentence_encoder
        self.domain_encoders  = nn.ModuleDict(domain_encoders)
        self.config           = bert_config
        self.dropout          = nn.Dropout(dropout)
        self.domain_names     = list(domain_encoders.keys())
        H = bert_config.hidden_size

        self.interaction_layers = nn.ModuleDict({
            d: LatentInteraction(H, interaction_hidden) for d in self.domain_names
        })
        self.classifier = nn.Linear(interaction_hidden * len(self.domain_names), num_labels)
        self._init_weights()

    def _init_weights(self):
        std = self.config.initializer_range
        for layer in self.interaction_layers.values():
            nn.init.normal_(layer.linear.weight, std=std)
            nn.init.zeros_(layer.linear.bias)
        nn.init.normal_(self.classifier.weight, std=std)
        nn.init.zeros_(self.classifier.bias)

    def _mean_pool(self, seq, mask):
        m = mask.unsqueeze(-1).float()
        return (seq * m).sum(1) / m.sum(1).clamp(min=1)

    def forward(self, input_ids, attention_mask, token_type_ids, target_mask,
                domain_input_ids, domain_attention_masks, labels=None):
        sent_out = self.sentence_encoder(
            input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids
        )
        ctx_emb = self._mean_pool(self.dropout(sent_out[0]), target_mask)

        interactions, distances = [], {}
        for domain in self.domain_names:
            dom_out = self.domain_encoders[domain](
                domain_input_ids[domain],
                attention_mask=domain_attention_masks[domain],
            )
            basic_emb = self._mean_pool(self.dropout(dom_out[0]),
                                        domain_attention_masks[domain])

            distances[domain] = 1.0 - F.cosine_similarity(basic_emb, ctx_emb, dim=-1)
            interactions.append(self.dropout(
                self.interaction_layers[domain](basic_emb, ctx_emb)
            ))

        logits = self.classifier(torch.cat(interactions, dim=-1))

        all_dist      = torch.stack([distances[d] for d in self.domain_names], dim=-1)
        max_domain_idx = all_dist.argmax(dim=-1)

        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            return loss, logits, max_domain_idx
        return logits, max_domain_idx, distances


# ── Training utilities ────────────────────────────────────────────────────────

def evaluate_metaphor(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            labels = batch['label'].to(device)
            inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else
                      {dk: dv.to(device) for dk, dv in v.items()}
                      for k, v in batch.items() if k != 'label'}
            loss, logits, _ = model(**inputs, labels=labels)
            total_loss += loss.item()
            all_preds.extend(logits.argmax(-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    return total_loss / max(len(loader), 1), f1


def train_metaphor_model(model, train_loader, val_loader, device,
                         num_epochs=10, warmup_epochs=2, lr=2e-5):
    total_steps  = len(train_loader) * num_epochs
    warmup_steps = len(train_loader) * warmup_epochs

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    best_f1, best_state = 0.0, None

    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for batch in tqdm.tqdm(train_loader, desc=f'Epoch {epoch}/{num_epochs}'):
            labels = batch['label'].to(device)
            inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else
                      {dk: dv.to(device) for dk, dv in v.items()}
                      for k, v in batch.items() if k != 'label'}
            optimizer.zero_grad()
            loss, _, _ = model(**inputs, labels=labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        train_loss = epoch_loss / len(train_loader)
        val_loss, val_f1 = evaluate_metaphor(model, val_loader, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)

        print(f'Epoch {epoch} | train_loss={train_loss:.4f} '
              f'| val_loss={val_loss:.4f} | val_F1={val_f1:.4f} '
              f'| lr={scheduler.get_last_lr()[0]:.2e}')

        if val_f1 > best_f1:
            best_f1  = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state:
        model.load_state_dict(best_state)
        print(f'Best val F1: {best_f1:.4f}')
    return model, history


# ── Initialization ────────────────────────────────────────────────────────────

tokenizer   = add_pos_tokens(tokenizer)
bert_config = AutoConfig.from_pretrained('google-bert/bert-base-german-cased')

sentence_bert = AutoModel.from_pretrained('google-bert/bert-base-german-cased')
sentence_bert.resize_token_embeddings(len(tokenizer))

# Extract BertModel encoders from domain-fine-tuned BertForMaskedLM models
domain_bert_encoders = {}
for domain, mlm_model in domain_models.items():
    encoder = mlm_model.bert if hasattr(mlm_model, 'bert') else mlm_model
    encoder.resize_token_embeddings(len(tokenizer))
    domain_bert_encoders[domain] = encoder

metaphor_model = MetaphorBERT(
    sentence_encoder=sentence_bert,
    domain_encoders=domain_bert_encoders,
    bert_config=bert_config,
    num_labels=2,
    dropout=0.2,
    interaction_hidden=256,
).to(device)

n_params = sum(p.numel() for p in metaphor_model.parameters())
print(f'MetaphorBERT | domains={list(domain_bert_encoders.keys())} | params={n_params:,}')


# ── DataLoaders ───────────────────────────────────────────────────────────────

domain_names = list(domain_bert_encoders.keys())

train_dataset = MetaphorDataset(anthro_train, tokenizer, domain_names, label_col='label')
val_dataset   = MetaphorDataset(anthro_val,   tokenizer, domain_names, label_col='label')
test_dataset  = MetaphorDataset(anthro_test,  tokenizer, domain_names, label_col='label')

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  collate_fn=collate_metaphor)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, collate_fn=collate_metaphor)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False, collate_fn=collate_metaphor)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

In [ ]:
# ── Train MetaphorBERT ────────────────────────────────────────────────────────

metaphor_model, metaphor_history = train_metaphor_model(
    model=metaphor_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=10,
    warmup_epochs=2,
    lr=2e-5,
)

torch.save(metaphor_model.state_dict(), f'{metaphors_dir}/metaphor_bert_best.pt')
print('Model saved.')

# ── Test evaluation ───────────────────────────────────────────────────────────

test_loss, test_f1 = evaluate_metaphor(metaphor_model, test_loader, device)
print(f'Test loss={test_loss:.4f} | Test F1={test_f1:.4f}')

# Per-sample output: prediction + dominant metaphor domain
metaphor_model.eval()
results = []
with torch.no_grad():
    for batch in test_loader:
        labels = batch['label'].to(device)
        inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else
                  {dk: dv.to(device) for dk, dv in v.items()}
                  for k, v in batch.items() if k != 'label'}
        logits, max_domain_idx, _ = metaphor_model(**inputs)
        preds    = logits.argmax(-1).cpu().tolist()
        dom_names = [domain_names[i] for i in max_domain_idx.cpu().tolist()]
        for pred, label, dom in zip(preds, labels.cpu().tolist(), dom_names):
            results.append({'prediction': pred, 'label': label, 'dominant_domain': dom})

results_df = pd.DataFrame(results)
print(results_df.head(20))
print('\nDomain distribution in metaphorical predictions:')
print(results_df[results_df['prediction'] == 1]['dominant_domain'].value_counts())

In [ ]:
epochs = range(1, len(metaphor_history['train_loss']) + 1)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

ax1.plot(epochs, metaphor_history['train_loss'], marker='o', color='steelblue')
ax1.set_title('Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, metaphor_history['val_loss'], marker='o', color='coral')
ax2.set_title('Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.grid(True, alpha=0.3)

ax3.plot(epochs, metaphor_history['val_f1'], marker='o', color='seagreen')
ax3.set_title('Validation F1')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('F1')
ax3.set_ylim(0, 1)
ax3.grid(True, alpha=0.3)

plt.suptitle('MetaphorBERT Training', fontsize=13)
plt.tight_layout()
plt.savefig(f'{plots_dir}/metaphor_bert_training.png', dpi=150)
plt.show()